In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import os
import shutil
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# SETUP & CONFIGURATION
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# Intercepting Hugging Face to prevent Jupyter crashes
try:
    import huggingface_hub.utils._progress
    import tqdm.std
    huggingface_hub.utils._progress.tqdm = tqdm.std.tqdm
except Exception:
    pass

# Updating these folders to match where your images are stored
RAW_IMAGES_FOLDER = "./haryana village house/"       
CLEAN_IMAGE_FOLDER = "./02_RAW_IMAGES/"   

os.makedirs(RAW_IMAGES_FOLDER, exist_ok=True)
os.makedirs(CLEAN_IMAGE_FOLDER, exist_ok=True)

print("Loading CLIP Model OFFLINE from local folder...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware Acceleration: {device.upper()}")

# Loading the model directly from your hard drive!
model_id = "./local_clip_model" 
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

# UPDATED AI GATEKEEPER PROMPTS
CATEGORIES = [
    "a photograph focusing on a single house or a specific room inside one house", # Index 0: What we want
    "a wide street view showing multiple different houses, a village lane, or a neighborhood", # Index 1: The NEW junk to drop
    "a scanned document, blank page, text, close-up texture, or other unrelated items" # Index 2: The OLD junk to drop
]

# Supported image extensions (ignores .txt or system files)
VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

# MAIN AI FILTER PIPELINE
def process_images():
    print("\nStarting Image Filtering...")
    total_processed = 0
    total_saved = 0
    
    # Looping through every file in the raw images folder
    for image_name in os.listdir(RAW_IMAGES_FOLDER):
        
        # Skiping files that aren't images
        if not image_name.lower().endswith(VALID_EXTENSIONS):
            continue
            
        image_path = os.path.join(RAW_IMAGES_FOLDER, image_name)
        total_processed += 1
        
        try:
            # Loading the image for the AI to see
            image = Image.open(image_path).convert("RGB")
            
            # Running the CLIP AI Gatekeeper
            inputs = processor(text=CATEGORIES, images=image, return_tensors="pt", padding=True).to(device)
            
            with torch.no_grad():
                outputs = model(**inputs)
                probs = outputs.logits_per_image.softmax(dim=-1)
            
            # Grabing the probabilities
            prob_single_house = probs[0][0].item()
            prob_multiple_houses = probs[0][1].item()
            
            # 1. It must be at least 60% confident it fits the first prompt.
            # 2. The "single house" score MUST beat the "multiple houses" score.
            if prob_single_house > 0.60 and prob_single_house > prob_multiple_houses:
                save_path = os.path.join(CLEAN_IMAGE_FOLDER, image_name)
                
                # Using shutil.copy preserves the exact original file quality without compressing it
                shutil.copy(image_path, save_path)
                total_saved += 1
                
                print(f"  [+] Kept (Single House): {image_name} (Confidence: {prob_single_house:.1%})")
            else:
                # If it's dropped, print out WHY it was dropped to help you monitor it
                if prob_multiple_houses > prob_single_house:
                    reason = "Multiple Houses Detected"
                else:
                    reason = "Junk/Document Detected"
                    
                print(f"  [-] Dropped ({reason}): {image_name}")
                    
        except Exception as e:
            print(f"Error reading {image_name}: {e}")

    print("\n" + "="*40)
    print("PIPELINE COMPLETE")
    print(f"Total raw images processed: {total_processed}")
    print(f"Total valid photos saved: {total_saved}")
    print(f"Removed {total_processed - total_saved} useless images!")
    print("="*40)

if __name__ == "__main__":
    process_images()

Loading CLIP Model OFFLINE from local folder...
Hardware Acceleration: CPU


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Starting Image Filtering...
  [-] Dropped (Multiple Houses Detected): 103995754.jpg
  [+] Kept (Single House): 103995775.jpg (Confidence: 77.8%)
  [+] Kept (Single House): 10_1501494420.jpg (Confidence: 100.0%)
  [-] Dropped (Multiple Houses Detected): 1499401242-village.jpg
  [+] Kept (Single House): 1_1501492211.jpg (Confidence: 99.8%)
  [+] Kept (Single House): 455728400O-1767614455305.jpg (Confidence: 98.3%)
  [+] Kept (Single House): 466949-whatsappimage2024-08-09at43621pm.webp (Confidence: 66.0%)
  [+] Kept (Single House): 4ddc35cb2565e989268296206cb5ae6f.jpg (Confidence: 97.6%)
  [+] Kept (Single House): 4df4fd7ff4cd887975aff2bb9f25c23a.jpg (Confidence: 94.0%)
  [+] Kept (Single House): 660-113.jpg (Confidence: 99.9%)
  [-] Dropped (Multiple Houses Detected): 70162690.jpg
  [+] Kept (Single House): 779122236M-1781757230948.webp (Confidence: 99.6%)
  [+] Kept (Single House): 828484-fowbqnpdrb-1486144710.jpg (Confidence: 99.8%)
  [-] Dropped (Junk/Document Detected): ACg8ocKevpnp

C:\Users\Harsh Datt\anaconda3\Lib\site-packages\PIL\Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  [-] Dropped (Junk/Document Detected): icc-womens-t20-world-cup-2026-semi-final-1-6753651837111181- (1).png
  [-] Dropped (Junk/Document Detected): icc-womens-t20-world-cup-2026-semi-final-1-6753651837111181- (2).png
  [-] Dropped (Junk/Document Detected): icc-womens-t20-world-cup-2026-semi-final-1-6753651837111181-.png
  [+] Kept (Single House): image (1).jpeg (Confidence: 85.3%)
  [-] Dropped (Junk/Document Detected): image (1).png
  [+] Kept (Single House): image (10).jpeg (Confidence: 92.3%)
  [+] Kept (Single House): image (11).jpeg (Confidence: 99.6%)
  [+] Kept (Single House): image (12).jpeg (Confidence: 98.0%)
  [+] Kept (Single House): image (13).jpeg (Confidence: 77.2%)
  [+] Kept (Single House): image (14).jpeg (Confidence: 97.1%)
  [+] Kept (Single House): image (15).jpeg (Confidence: 99.3%)
  [+] Kept (Single House): image (16).jpeg (Confidence: 99.9%)
  [+] Kept (Single House): image (17).jpeg (Confidence: 96.0%)
  [+] Kept (Single House): image (18).jpeg (Confidence: 8

In [1]:
import os
import shutil
import torch
import cv2
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# Intercepting Hugging Face to prevent Jupyter crashes
try:
    import huggingface_hub.utils._progress
    import tqdm.std
    huggingface_hub.utils._progress.tqdm = tqdm.std.tqdm
except Exception:
    pass

RAW_IMAGES_FOLDER = "./02_RAW_IMAGES/"       
CLEAN_IMAGE_FOLDER = "./02_CLEAN_IMAGES/"   

os.makedirs(RAW_IMAGES_FOLDER, exist_ok=True)
os.makedirs(CLEAN_IMAGE_FOLDER, exist_ok=True)

# THRESHOLDS FOR PHYSICAL GATES
MIN_THUMBNAIL_SIZE = 128 # Drops anything smaller than 128x128. Keeps anything larger for padding.
BLUR_THRESHOLD = 50.0    # If the sharpness score is below this, it's considered blurry.

print("Loading CLIP Model OFFLINE from local folder...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware Acceleration: {device.upper()}")

model_id = "./local_clip_model" 
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

# ==========================================
# 2. UPDATED AI GATEKEEPER PROMPTS
# ==========================================
CATEGORIES = [
    "a photograph focusing on a single house or a specific room inside one house", # Index 0: TARGET
    "a photo collage containing multiple pictures stitched together in a grid",    # Index 1: DROP (Collages)
    "a wide street view showing multiple different houses, a village lane",        # Index 2: DROP (Multiple)
    "a scanned document, blank page, text, close-up texture, or unrelated items"   # Index 3: DROP (Junk)
]

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

# ==========================================
# 3. HELPER FUNCTION: BLUR DETECTION
# ==========================================
def check_sharpness(pil_img):
    """Converts image to grayscale and measures edge sharpness. Higher = Sharper."""
    # Convert PIL Image to an OpenCV Grayscale array
    cv_img = np.array(pil_img.convert('L'))
    # Calculate the variance of the Laplacian (standard blur detection math)
    sharpness_score = cv2.Laplacian(cv_img, cv2.CV_64F).var()
    return sharpness_score

# ==========================================
# 4. MAIN FILTER PIPELINE
# ==========================================
def process_images():
    print("\nStarting Advanced Image Filtering...")
    total_processed = 0
    total_saved = 0
    
    for image_name in os.listdir(RAW_IMAGES_FOLDER):
        
        if not image_name.lower().endswith(VALID_EXTENSIONS):
            continue
            
        image_path = os.path.join(RAW_IMAGES_FOLDER, image_name)
        total_processed += 1
        
        try:
            image = Image.open(image_path).convert("RGB")
            
            # --- GATE 1: THE SIZE CHECK ---
            width, height = image.size
            if width < MIN_THUMBNAIL_SIZE and height < MIN_THUMBNAIL_SIZE:
                print(f"  [-] Dropped (Thumbnail): {image_name} [{width}x{height}]")
                continue # Skip the rest of the checks to save time
                
            # --- GATE 2: THE BLUR CHECK ---
            sharpness = check_sharpness(image)
            if sharpness < BLUR_THRESHOLD:
                print(f"  [-] Dropped (Blurry): {image_name} [Score: {sharpness:.1f}]")
                continue # Skip the AI check to save GPU memory
            
            # --- GATE 3: THE AI CHECK (Collages & Content) ---
            inputs = processor(text=CATEGORIES, images=image, return_tensors="pt", padding=True).to(device)
            
            with torch.no_grad():
                outputs = model(**inputs)
                probs = outputs.logits_per_image.softmax(dim=-1)
            
            prob_single = probs[0][0].item()
            prob_collage = probs[0][1].item()
            prob_multiple = probs[0][2].item()
            prob_junk = probs[0][3].item()
            
            # It must be confident in the single house, AND it must beat all 3 junk categories
            if prob_single > 0.50 and prob_single > max(prob_collage, prob_multiple, prob_junk):
                save_path = os.path.join(CLEAN_IMAGE_FOLDER, image_name)
                shutil.copy(image_path, save_path)
                total_saved += 1
                print(f"  [+] Kept: {image_name} (Confidence: {prob_single:.1%}, Sharpness: {sharpness:.1f})")
            else:
                # Figure out exactly which junk category triggered the drop
                highest_junk = max(prob_collage, prob_multiple, prob_junk)
                if highest_junk == prob_collage:
                    reason = "Collage Detected"
                elif highest_junk == prob_multiple:
                    reason = "Multiple Houses"
                else:
                    reason = "Junk/Document"
                    
                print(f"  [-] Dropped ({reason}): {image_name}")
                    
        except Exception as e:
            print(f"Error reading {image_name}: {e}")

    print("\n" + "="*40)
    print("PIPELINE COMPLETE")
    print(f"Total raw images processed: {total_processed}")
    print(f"Total valid photos saved: {total_saved}")
    print(f"Removed {total_processed - total_saved} useless images!")
    print("="*40)

if __name__ == "__main__":
    process_images()

Loading CLIP Model OFFLINE from local folder...
Hardware Acceleration: CPU


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Starting Advanced Image Filtering...
  [+] Kept: 10_1501494420.jpg (Confidence: 98.5%, Sharpness: 1112.6)
  [+] Kept: 1_1501492211.jpg (Confidence: 86.9%, Sharpness: 1079.9)
  [+] Kept: 455728400O-1767614455305.jpg (Confidence: 98.5%, Sharpness: 997.7)
  [+] Kept: 466949-whatsappimage2024-08-09at43621pm.webp (Confidence: 74.5%, Sharpness: 334.2)
  [+] Kept: 4ddc35cb2565e989268296206cb5ae6f.jpg (Confidence: 97.4%, Sharpness: 8149.4)
  [+] Kept: 660-113.jpg (Confidence: 98.2%, Sharpness: 4115.1)
  [+] Kept: 779122236M-1781757230948.webp (Confidence: 99.1%, Sharpness: 1929.4)
  [+] Kept: 828484-fowbqnpdrb-1486144710.jpg (Confidence: 98.8%, Sharpness: 2139.1)
  [+] Kept: approach-entrance.jpg (Confidence: 82.5%, Sharpness: 1006.6)
  [+] Kept: architecture-aNew-18.jpg (Confidence: 88.6%, Sharpness: 607.9)
  [+] Kept: b00bb7484a4fa9ab80a87d0c71a9750c.jpg (Confidence: 62.4%, Sharpness: 1075.3)
  [+] Kept: banni_khera.jpg (Confidence: 74.2%, Sharpness: 1023.2)
  [+] Kept: bhullar-villa-mallek

In [2]:
pip uninstall opencv-python -y

Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
